## Common imports

In [1]:
from src.config.data_contract_preprocessing import *
from src.config.data_contract_postprocessing import *
from src.config.data_contract_selection import *
from src.preprocessing.clinical_preprocessing import *
from src.preprocessing.protein_preprocessing import *
from src.selection.clinical_selection import *
from src.selection.protein_selection import *
from src.selection.build_dataset import *

## Define directory and file paths

In [2]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_DIR = PROJECT_ROOT / "datasets_final"
INITIAL_AUTOIMMUNE_FILE = "Protein.xlsx"
INITIAL_CLINICAL_FILE = "Clinical.xlsx"
from pathlib import Path

print(Path.cwd())
print(Path(DATA_DIR))
print(Path(DATA_DIR) / INITIAL_AUTOIMMUNE_FILE)
print(Path(DATA_DIR) / INITIAL_CLINICAL_FILE)

C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\notebooks\run_data_preparation_pipeline
C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets_final
C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets_final\Protein.xlsx
C:\Users\barbs\COMPUTER SCIENCE\FH CAMPUS\6_COURSES SS 26\KI Wahlfach Projekt\Source Code_Main_1\RA-Map-EDA\datasets_final\Clinical.xlsx


## Load datasets into dataframes (protein and clinical)

In [3]:
data = load_all_sheets()
df_clinical = data["df_clinical"]
df_steroids = data["df_steroids"]
df_meds = data["df_meds"]
df_protein = data["df_Samples"]

Loading clinical...
Loading protogen...
Done loading


## Full pre-processing clinical

In [4]:
# Remove non-RA patients
df_clinical_ra = filter_ra_cohort(df_clinical)
# Remove all cohort columns
df_clinical_no_cohort = drop_cohort_columns(df_clinical_ra, CLINICAL_CONTRACT)
# Codify all binary
df_clinical_binary = clean_binary_columns(df_clinical_no_cohort, CLINICAL_CONTRACT)
# Drop columns I know are not useful
df_clinical_dropped = drop_columns(df_clinical_binary, ["Hep B serology wk 9 (IU/mL)", "ORAL.STEROIDS.3M", "IM.STEROIDS.3MONTHS"])
# Adjust the column Remission month
df_clinical_remission = clean_remission_month(df_clinical_dropped, col="Remission month")
# Clean remaining numerical data
df_clinical_numeric = clean_clinical_numeric(df_clinical_remission,CLINICAL_CONTRACT,verbose=True)
# Postprocess missing values
df_clinical_post = postprocess_missing_values(df_clinical_numeric, CLINICAL_CONTRACT_POST)
# Normalise numeric columns
df_clinical_imputed = impute_numeric_values(df_clinical_post,CLINICAL_CONTRACT_POST,verbose=True)
# Export to .csv
df_clinical_imputed.to_csv("../../datasets_final/Clinical_Clean.csv", index=False)
# clean steroid data
df_steroids_clean = clean_event_level_data(df_steroids, STEROIDS_CONTRACT)
# Clean med data
df_meds_clean = clean_event_level_data(df_meds, MEDS_CONTRACT)
# Aggregate event-level data
df_steroids_agg = aggregate_steroids(df_steroids_clean)
df_meds_agg = aggregate_meds(df_meds_clean)
# Merge all clinical data into one dataframe
df_final_merged = merge_patient_datasets(df_clinical_imputed, df_steroids_agg, df_meds_agg)
# Export the merged dataset to .csv
df_final_merged.to_csv("../../datasets_final/Clinical_Clean_Merged.csv", index=False)
# Normalise through log-transform of all merged clinical data
df_clinical_log, numeric_cols = preprocess_log_pipeline(df_final_merged, CLINICAL_CONTRACT_POST, STEROID_FEATURES_CONTRACT_POST, MEDS_FEATURES_CONTRACT_POST)
# Export the log-transformed data to .csv
df_clinical_log.to_csv("../../datasets_final/Clinical_Clean_Merged_Log.csv", index=False)
# Use z-score to normalise log-transformed data
df_clinical_standardised, scaler = preprocess_zscore_pipeline(df_clinical_log, CLINICAL_CONTRACT_POST, STEROID_FEATURES_CONTRACT_POST,MEDS_FEATURES_CONTRACT_POST)
# Export to .csv
df_clinical_standardised.to_csv("../../datasets_final/Clinical_Clean_Merged_Log_Standardised.csv", index=False)



===== RA COHORT FILTER =====
Before filtering: 327 rows
After filtering:  275 rows
Removed (non-RA / other cohorts): 52


===== COHORT COLUMN REMOVAL =====
Removed 6 columns:
['Study', 'Region', 'Hub', 'REGION_HUB', 'RACE', 'vaccine centre']
Remaining shape: (275, 61)


===== COLUMN DROP =====
Dropped columns: ['Hep B serology wk 9 (IU/mL)', 'ORAL.STEROIDS.3M', 'IM.STEROIDS.3MONTHS']
Remaining shape: (275, 58)


Numeric preprocessing completed
Shape: (275, 59)

Top missingness:
DAS28.18M           0.883636
TOTAL.TENDER.9M     0.549091
SDAI.12M            0.443636
DAS28.9M            0.425455
CRP.9M              0.421818
TOTAL.SWOLLEN.9M    0.421818
DAS28.12M           0.389091
BASOPHILS.6M        0.200000
HAQ.6M              0.196364
PLT.6M              0.192727
dtype: float64

===== NUMERIC IMPUTATION =====
Strategy: median
Imputed columns: 47
Shape: (275, 59)

[LOG] applying log1p to 36 columns
[ZSCORE] scaling 57 columns


## Full pre-processing protein

In [5]:
# Remove non-RA patients from the dataset
df_protein_ra = filter_ra_only(df_protein)
# Transpose the matrix
df_protein_transposed = transpose_autoimmune_data(df_protein_ra)
# Split set into BL and M6 (we will not do anything with the M6 data)
df_protein_bl, df_protein_m6 = split_bl_m6(df_protein_transposed)
# Normalise BL data using log transformation to reduce skewness and make the distribution more symmetric
df_protein_log = log_normalise(df_protein_bl)
# Export the BL normalised data sheet
df_protein_log.to_csv("../../datasets_final/Protein_Clean_BL_Log.csv", index=False)
# Standardise the log-transformed data by transforming to z-scores to have a mean of 0 and standard deviation of 1
df_protein_log_standardised = standardise_data(df_protein_log)
# Export the BL standardised data sheet
df_protein_log_standardised.to_csv("../../datasets_final/Protein_Clean_BL_Log_Standardised.csv", index=False)

Removed 93 VAC columns
Remaining dataset shape: (163, 504)
Transposition complete
Shape: (500, 163)
Index preview: Index(['TAC1241_BL', 'TAC1241_M6', 'TAC1147_BL', 'TAC1147_M6', 'TAC1094_BL'], dtype='str', name='Patient_Timepoint')
BL shape: (265, 163)
M6 shape: (235, 163)
Log normalisation complete
Shape: (265, 163)
Standardisation complete
Shape: (265, 163)


## Choose Clinical Features based on those viewed in the literature as most relevant (not variance-report based)
Variance is difficult to establish in clinical data, because not all values (e.g., age, symptom duration) should be log transformed. Therefore, they will show up as having high variance, but will not contribute biologically to the clustering exercise

In [6]:
df_clinical_selected = apply_clinical_selection_contract(df_clinical_standardised,CLINICAL_CONTRACT_SELECTION)
df_clinical_selected.to_csv("../../datasets_final/Clinical_Clean_Merged_Log_Standardised_Selected.csv", index=False)


===== CLINICAL CONTRACT SELECTION =====
Total selected columns: 21
Missing columns: 0
Final shape: (275, 21)



## Choose Protein Features

Choose 50 proteins with highest variance

In [7]:
# Select 50 features with higher variance (Use the log transformed dataset)
df_protein_var50, variance_rank = select_top_variance_features(df_protein_log,top_n=50)
df_protein_var50.to_csv("../../datasets_final/Protein_Clean_BL_Log_Selected_Variance50.csv", index=False)


===== VARIANCE SELECTION =====
Input features: 163
Selected features: 50



Choose proteins that most highly correlate with the binary value remission_event (use the log transformed protein set to calculate correlation, and later switch to the z-score set for final export)
Absolute - loss of direction (negative, positive)
Negative correlation = higher protein levels have lower chance of remission
Positive correlation = higher protein levels have higher chance of remission

In [8]:
df_protein_added = add_patient_id(df_protein_log)
df_remission = df_final_merged[["Patient_ID", "remission_event"]].copy()
df_protein_outcome = df_protein_added.merge(df_remission,on="Patient_ID",how="inner")
#How strongly each protein is associated with remission (binary outcome). A ranked list of proteins by predictive signal for remission
#Higher absolute value = stronger association - losing direction positive or negative (we don't need direction at this point)
#Positive sign = higher protein value linked to remission
#Negative sign = higher protein value linked to non-remission
df_protein_rank = rank_proteins_by_remission(df_protein_outcome)
top_proteins = df_protein_rank.abs().sort_values(ascending=False).head(50)
df_protein_selected = df_protein_outcome[["Patient_ID"] + top_proteins.index.tolist()]
df_protein_selected.to_csv("../../datasets_final/Protein_Clean_BL_Log_Selected_CorrelationRemission.csv", index=False)


Choose proteins that (according to the literature) are biologically relevant to RA
Strong RA-relevant core biology:
Cytokines / inflammation: IL6, IL1B, IL10, IL12B
Autoimmunity / immune signaling: MS4A1 (B cells), PADI4
ECM / tissue destruction: FN1, TNC, SPP1, BGN
Innate immune / neutrophils: LTF, CXCL5, PRTN3, ELANE, CTSG
Autoimmune RA hallmark proteins: SPP1, CALR, IGFBP family
Structural damage / synovial activity: VIM, FN1, LMNA

In [9]:
RA_LITERATURE_CORE = [
    "IL6","IL1B","IL10","IL12B",
    "MS4A1","PADI4",
    "FN1","TNC","SPP1","BGN","IGF1","IGFBP2","IGFBP6",
    "LTF","PRTN3","CXCL5",
    "CALR","VIM","HSPD1","LMNA"
]

df_protein_literature = select_ra_literature_proteins(df_protein_log_standardised, RA_LITERATURE_CORE, id_col="Patient_ID")
df_protein_literature.to_csv("../../datasets_final/Protein_Clean_BL_Log_Standardised_Selected_Literature.csv", index=False)

## Build final ML-Ready Datasets
1. Standardise dataframe format to get them ready for combining datasets
2. Ensure all are in z-score format
2. Combine clinical with protein - variance50
3. Combine clinical with protein - highly correlated to remission
4. Combine clinical with protein - biologically meaningful according to literature
5. Export as separate .csv Files

In [10]:
# Standardise all dataframes that are needed for set-building to ensure each has a Patient_ID or Patient_Timeframe column as well as the necessary columns with feature names

# Clinical sets
# Z-Score
df_clinical_literature_zscore = standardize_patient_column(df_clinical_selected)

# Protein sets
# Log-transformed
df_protein_var = standardize_patient_column(df_protein_var50)
df_protein_corr = standardize_patient_column(df_protein_selected)

# Z-Score
df_protein_literature_zscore = standardize_patient_column(df_protein_literature)
df_protein_log_zscore = standardize_patient_column(df_protein_log_standardised)


# Ensure selected feature values are in z-format ready for ML

In [11]:
# Variance based proteins (get the z-scores)
df_protein_variance_zscore, _ = select_features_from_zscore(
    df_protein_log_zscore,
    df_protein_var
)

# Correlation to remission based protein (get the z-scores)
df_protein_correlation_remission_zscore, _ = select_features_from_zscore(
    df_protein_log_zscore,
    df_protein_corr
)


## Combine:
1. Clinical z-score with protein variance52 z-score
2. Clinical z-score with protein correlation with remission z-score
3. Clinical z-score with protein literature z-score

In [12]:
# Manual cleanup of ID columns if needed
df_clinical_literature_zscore = df_clinical_literature_zscore.drop(columns=["Digest", "Patient_Timepoint"], errors="ignore")

df_protein_variance_zscore = df_protein_variance_zscore.drop(columns=["Digest", "Patient_Timepoint"], errors="ignore")
df_protein_variance_zscore = df_protein_variance_zscore.loc[:, ~df_protein_variance_zscore.columns.duplicated()]

df_protein_correlation_remission_zscore = df_protein_correlation_remission_zscore.loc[:, ~df_protein_correlation_remission_zscore.columns.duplicated()]

df_protein_literature_zscore = df_protein_literature_zscore.drop(columns=["Digest", "Patient_Timepoint"], errors="ignore")
df_protein_literature_zscore = df_protein_literature_zscore[["Patient_ID"] + [c for c in df_protein_literature_zscore.columns if c != "Patient_ID"]]

# Combine clinical with protein and export to .csv
df_ml_variance = df_clinical_literature_zscore.merge(
    df_protein_variance_zscore,
    on="Patient_ID",
    how="inner"
)

df_ml_correlation = df_clinical_literature_zscore.merge(
    df_protein_correlation_remission_zscore,
    on="Patient_ID",
    how="inner"
)

df_ml_literature = df_clinical_literature_zscore.merge(
    df_protein_literature_zscore,
    on="Patient_ID",
    how="inner"
)

df_ml_variance.to_csv("../../datasets_final/ml_ready/ml_variance.csv", index=False)
df_ml_correlation.to_csv("../../datasets_final/ml_ready/ml_correlation.csv", index=False)
df_ml_literature.to_csv("../../datasets_final/ml_ready/ml_literature.csv", index=False)